In [36]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
    size_adjusted_power_comparison,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = False
_AUGMENTED_PARAM = 'x_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.05
_MC_SAMPLES = 1000
_MC_ALPHA = 0.05
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)


In [37]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83   0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [38]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [39]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [40]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [41]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: False
Augmented measurement equation: OutGap
Augmented coefficient: x_coef
Monte Carlo replications: 1000
Noise Covariance:
 [[0.604 0.    0.   ]
 [0.    0.839 0.   ]
 [0.    0.    0.039]]


In [42]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 1000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,17.381,0.002,0.219,0.001,1000,992,0.992,0.003,0.984,0.996
1,Infl,27.144,0.000,0.260,0.000,1000,1000,1.000,0.000,0.996,1.000
2,Rate,15.366,0.006,0.221,0.001,1000,972,0.972,0.005,0.960,0.981


In [43]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 1000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.125,2.450,0.58,0.002,0.071,0.009,1000,33,0.033,0.006,0.024,0.046,3.0,200,4
1,cov_identity,16.541,298.632,0.00,0.061,2.964,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


In [44]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-1.125,-0.051,1.563,-0.730,0.428,0.007,0.048,0.002,0.004,0.031,0.010,0.000,1000,115,0.115,0.010,0.097,0.136
1,OutGap,x,-0.432,-0.155,0.193,-2.227,0.108,0.029,0.006,0.002,0.001,0.031,0.006,0.001,1000,606,0.606,0.015,0.575,0.636
2,OutGap,r,-0.554,-0.018,1.981,-0.260,0.494,0.005,0.062,0.002,0.007,0.031,0.009,0.000,1000,57,0.057,0.007,0.044,0.073
3,Infl,Pi,-0.918,-0.035,1.772,-0.501,0.455,0.006,0.058,0.002,0.004,0.032,0.009,0.000,1000,93,0.093,0.009,0.077,0.113
4,Infl,x,0.079,0.025,0.222,0.352,0.472,0.006,0.007,0.002,0.001,0.033,0.009,0.000,1000,74,0.074,0.008,0.059,0.092
5,Infl,r,-0.461,-0.013,2.245,-0.186,0.482,0.005,0.074,0.002,0.008,0.033,0.009,0.000,1000,49,0.049,0.007,0.037,0.064
6,Rate,Pi,-0.191,-0.040,0.340,-0.574,0.467,0.006,0.010,0.002,0.001,0.031,0.010,0.000,1000,93,0.093,0.009,0.077,0.113
7,Rate,x,0.023,0.038,0.043,0.538,0.454,0.006,0.001,0.002,0.000,0.032,0.009,0.000,1000,76,0.076,0.008,0.061,0.094
8,Rate,r,-0.552,-0.089,0.429,-1.267,0.305,0.013,0.014,0.002,0.001,0.031,0.009,0.000,1000,244,0.244,0.014,0.218,0.272


In [45]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-3.902,-0.310,0.842,-4.620,0.001,0.099,0.026,0.002,0.001,0.028,0.000,0.001,1000,998,0.998,0.001,0.993,0.999
1,OutGap,x,-0.524,-0.342,0.101,-5.147,0.000,0.120,0.003,0.002,0.000,0.029,0.000,0.001,1000,1000,1.000,0.000,0.996,1.000
0,OutGap,r,1.535,0.057,1.876,0.811,0.448,0.006,0.044,0.002,0.006,0.023,0.009,0.000,1000,60,0.060,0.008,0.047,0.076
5,Infl,Pi,-0.261,-0.017,1.003,-0.241,0.486,0.005,0.032,0.002,0.001,0.032,0.009,0.000,1000,53,0.053,0.007,0.041,0.069
4,Infl,x,0.002,0.003,0.122,0.038,0.507,0.005,0.004,0.002,0.000,0.032,0.009,0.000,1000,50,0.050,0.007,0.038,0.065
3,Infl,r,-0.510,-0.016,2.128,-0.227,0.485,0.006,0.069,0.002,0.007,0.033,0.009,0.000,1000,59,0.059,0.007,0.046,0.075
8,Rate,Pi,-0.026,-0.011,0.193,-0.154,0.513,0.005,0.006,0.002,0.000,0.031,0.009,0.000,1000,42,0.042,0.006,0.031,0.056
7,Rate,x,0.010,0.028,0.023,0.393,0.484,0.006,0.001,0.002,0.000,0.031,0.009,0.000,1000,72,0.072,0.008,0.058,0.090
6,Rate,r,-0.586,-0.100,0.406,-1.418,0.267,0.015,0.013,0.002,0.001,0.032,0.009,0.000,1000,299,0.299,0.014,0.271,0.328


In [46]:
print("Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"]).round(3)

Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.575,-2.699,-1.125,-1.125,-0.0,0.0,0.0,0.031,0.024,0.048,0.048,0.0,0.0,0.0
1,OutGap,x,0.035,-0.467,-0.432,-0.432,0.0,0.0,0.0,0.004,0.003,0.006,0.006,0.0,0.0,0.0
2,OutGap,r,-0.183,-0.371,-0.554,-0.554,0.0,0.0,0.0,0.039,0.033,0.062,0.062,0.0,0.0,0.0
3,Infl,Pi,-0.035,-0.884,-0.918,-0.918,-0.0,0.0,0.0,0.013,0.056,0.058,0.058,0.0,0.0,0.0
4,Infl,x,0.002,0.077,0.079,0.079,0.0,0.0,0.0,0.002,0.007,0.007,0.007,0.0,0.0,0.0
5,Infl,r,-0.024,-0.438,-0.461,-0.461,0.0,0.0,0.0,0.017,0.073,0.074,0.074,0.0,0.0,0.0
6,Rate,Pi,0.006,-0.196,-0.191,-0.191,0.0,0.0,0.0,0.003,0.010,0.010,0.010,0.0,0.0,0.0
7,Rate,x,-0.000,0.023,0.023,0.023,-0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.010,-0.542,-0.552,-0.552,0.0,0.0,0.0,0.003,0.014,0.014,0.014,0.0,0.0,0.0


In [47]:
print("Innovation decomposition on raw predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"]).round(3)

Innovation decomposition on raw predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.869,-5.771,-3.902,-3.902,0.0,0.0,0.0,0.017,0.012,0.026,0.026,0.0,0.0,0.0
1,OutGap,x,0.198,-0.723,-0.524,-0.524,-0.0,0.0,0.0,0.002,0.002,0.003,0.003,0.0,0.0,0.0
2,OutGap,r,-0.664,2.199,1.535,1.535,-0.0,0.0,0.0,0.046,0.035,0.044,0.044,0.0,0.0,0.0
3,Infl,Pi,-0.017,-0.244,-0.261,-0.261,-0.0,0.0,0.0,0.007,0.032,0.032,0.032,0.0,0.0,0.0
4,Infl,x,-0.002,0.004,0.002,0.002,-0.0,0.0,0.0,0.001,0.004,0.004,0.004,0.0,0.0,0.0
5,Infl,r,-0.020,-0.490,-0.510,-0.510,0.0,0.0,0.0,0.016,0.069,0.069,0.069,0.0,0.0,0.0
6,Rate,Pi,0.005,-0.031,-0.026,-0.026,0.0,0.0,0.0,0.002,0.006,0.006,0.006,0.0,0.0,0.0
7,Rate,x,0.000,0.009,0.010,0.010,0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.008,-0.578,-0.586,-0.586,-0.0,0.0,0.0,0.003,0.013,0.013,0.013,0.0,0.0,0.0


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw so the original visual workflow remains available.


In [48]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()

## Diagnostics of the Augmented Model

The figures below still display the representative first draw. The scalar summaries reported in later cells are Monte Carlo averages.


### Marginal LR Test Conditional on $	heta_0$

The table below reports the Monte Carlo MLE summary for the LR test.


In [49]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,1.458,-2511.258,-1084.339,2853.839,0.0,0.002,6.188,0.837,11.75,0.0,1000,1000,1.0,0.0,0.996,1.0


In [50]:
res_mle

OptimizationResult(kind='mle', x=array([1.312394]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(1.3123940025746041), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(1067.399428705555), loglik=np.float64(-1067.399428705555), logprior=np.float64(0.0), logpost=np.float64(-1067.399428705555), nfev=16, nit=7, raw=  message: CONVERGENCE: REL

## Serial Autocorrelation Tests for the Augmented Model

The figure below uses the representative MCMC draw, while the printed table reports Monte Carlo MLE rejection frequencies.


In [51]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,2.057,0.349,0.077,0.009,1000,181,0.181,0.012,0.158,0.206
1,Infl,23.440,0.000,0.243,0.000,1000,1000,1.000,0.000,0.996,1.000
2,Rate,12.820,0.013,0.208,0.002,1000,937,0.937,0.008,0.920,0.950


In [52]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.125,2.450,0.58,0.002,0.071,0.009,1000,33,0.033,0.006,0.024,0.046,3.0,200,4
1,cov_identity,16.541,298.632,0.00,0.061,2.964,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.098,2.643,0.546,0.001,0.071,0.009,1000,38,0.038,0.006,0.028,0.052,3.0,200,4
1,cov_identity,1.127,38.574,0.001,0.007,0.411,0.000,1000,999,0.999,0.001,0.994,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


,test,n_replications,distance_ref,mc_se_distance_ref,distance_aug,mc_se_distance_aug,distance_improvement,mc_se_distance_improvement,stat_ref,mc_se_stat_ref,stat_aug,mc_se_stat_aug,stat_improvement,mc_se_stat_improvement,aug_closer_rate,aug_closer_rate_mc_se,aug_closer_ci_low,aug_closer_ci_high
0,mean_zero_hac,1000,0.125,0.002,0.098,0.001,0.027,0.001,2.450,0.071,2.643,0.071,-0.193,0.019,0.873,0.011,0.851,0.892
1,cov_identity,1000,16.541,0.061,1.127,0.007,15.414,0.062,298.632,2.964,38.574,0.411,260.058,2.838,1.000,0.000,0.996,1.000
